# FM Save Copilot

Upload your FM24 squad export and get a Director of Football briefing. Run each cell below in order (the ▶ button on the left of each cell, top to bottom) — no coding needed.

**This notebook only works inside Google Colab** (it uses Colab's file upload and secrets features).

In [ ]:
#@title Step 0: Setup (run this first) { display-mode: "form" }
import sys, subprocess, os

REPO_DIR = "/content/fm-save-copilot"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/laweh-dev/fm-save-copilot.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=True)
subprocess.run(["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from fm_copilot import parser, analyzer, report, tactics, config as config_module

print("Setup complete — ready to go.")

In [ ]:
#@title Step 1: Upload your squad export { display-mode: "form" }
from google.colab import files

print("Choose your FM24 squad HTML export...")
_uploaded = files.upload()
SQUAD_PATH = list(_uploaded.keys())[0]
print(f"\nUploaded: {SQUAD_PATH}")

In [ ]:
#@title Step 2: League context (optional) { display-mode: "form" }
add_league_context = False #@param {type:"boolean"}

LEAGUE_PATH = None
if add_league_context:
    from google.colab import files
    print("Choose your current-league HTML export...")
    _uploaded = files.upload()
    LEAGUE_PATH = list(_uploaded.keys())[0]
    print(f"\nUploaded: {LEAGUE_PATH}")
else:
    print("Skipped — no league benchmarking this run.")

In [ ]:
#@title Step 3: Configure your report { display-mode: "form" }
objective = "" #@param {type:"string"}
formation_override = "Auto-detect (recommended)" #@param ["Auto-detect (recommended)", "4-2-3-1", "4-3-3", "3-5-2", "3-4-3", "4-4-2", "3-4-2-1"]
tactical_direction = "Not specified" #@param ["Not specified", "Control Possession & High Press", "Gegenpress", "Low Block & Fast Counters", "Low Block & Waste Time", "Low Block & Direct Long Passing", "Tiki-Taka"]
report_type = "Free mode (no API key needed)" #@param ["Free mode (no API key needed)", "Full DoF narrative (needs an API key)"]

OBJECTIVE = objective or None
FORMATION = None if formation_override.startswith("Auto-detect") else formation_override
TACTIC_RAW = None if tactical_direction == "Not specified" else tactical_direction

print("Configuration set:")
print(f"  Objective: {OBJECTIVE or 'not specified'}")
print(f"  Formation: {FORMATION or 'auto-detect'}")
print(f"  Tactical direction: {TACTIC_RAW or 'not specified'}")
print(f"  Report type: {report_type}")

if add_league_context and not TACTIC_RAW:
    print("\n⚠️  League context needs a tactical direction — pick one above, or turn off")
    print("    league context in Step 2, before generating.")

In [ ]:
#@title Step 4: API key (only needed for Full DoF narrative) { display-mode: "form" }
if report_type.startswith("Full"):
    from google.colab import userdata
    try:
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("Found ANTHROPIC_API_KEY in Colab Secrets.")
    except Exception:
        print("No secret named ANTHROPIC_API_KEY found.\n")
        print("To add one:")
        print("  1. Click the key icon 🔑 in the left sidebar of this notebook.")
        print("  2. Click 'Add new secret'.")
        print("  3. Name: ANTHROPIC_API_KEY   Value: your Anthropic API key.")
        print("  4. Toggle 'Notebook access' on for this notebook.")
        print("  5. Re-run this cell.")
        print("\nYou only need to do this once — it's saved to your Google account,")
        print("not this notebook, and works in every future session.")
else:
    print("Free mode selected — no API key needed.")

In [ ]:
#@title Step 5: Generate report { display-mode: "form" }
from IPython.display import HTML, display
from google.colab import files as colab_files

tactical_style = None
if TACTIC_RAW:
    try:
        tactical_style = tactics.resolve_style_key(TACTIC_RAW)
    except ValueError as exc:
        print(f"[tactics] ERROR: {exc}")
        raise SystemExit

league_players = None
if add_league_context and LEAGUE_PATH:
    try:
        league_players = parser.parse_league(LEAGUE_PATH)
    except Exception as exc:
        print(f"[league] ERROR: {exc}")
        raise SystemExit

try:
    players = parser.parse_squad(SQUAD_PATH)
except Exception as exc:
    print(f"[parser] ERROR: {exc}")
    raise SystemExit

cfg = config_module.load_config()

try:
    analysis = analyzer.analyze(
        players, OBJECTIVE, FORMATION,
        tactical_style=tactical_style, league_players=league_players,
    )
except Exception as exc:
    print(f"[analyzer] ERROR: {exc}")
    raise SystemExit

OUT_PATH = "/content/report.html"
try:
    report.generate(analysis, players, OBJECTIVE, FORMATION, cfg, OUT_PATH)
except Exception as exc:
    print(f"[report] ERROR: {exc}")
    raise SystemExit

print("\n--- Preview ---\n")
display(HTML(filename=OUT_PATH))

print("\nDownloading report.html to your computer...")
colab_files.download(OUT_PATH)

---
Want to run another report (different squad, different tactic)? Just re-run from Step 1.